In [5]:
"""
PDF → YOLO layout detection → Vision LLM → Markdown
Processes all pages in parallel, preserves reading order via prompt.
"""

import os
import base64
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import fitz  # PyMuPDF
import numpy as np
from portkey_ai import Portkey
from doclayout_yolo import YOLOv10


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PDF_PATH        = r"..\\..\\Sample PDFs\OP-RAG.pdf"
YOLO_WEIGHTS    = "..\\..\\yolo\\doclayout_yolo_docstructbench_imgsz1024.pt"
OUTPUT_DIR      = "output_markdown_OP-RAG"
MODEL_NAME      = "gpt-5.4-mini"
MAX_WORKERS     = 16
YOLO_IMG_SIZE   = 1024
YOLO_CONF       = 0.20
RENDER_ZOOM     = 2.0
MAX_BLOCKS      = 40   # cap per page to avoid overloading the prompt


# ---------------------------------------------------------------------------
# Clients and thread-local state
# ---------------------------------------------------------------------------

client = Portkey(
    base_url=os.environ["PORTKEY_BASE_URL"],
    api_key=os.environ["PORTKEY_API_KEY"],
)

_thread_local = threading.local()

def get_yolo_model() -> YOLOv10:
    """Return a thread-local YOLO model instance (loaded once per thread)."""
    if not hasattr(_thread_local, "model"):
        _thread_local.model = YOLOv10(YOLO_WEIGHTS)
    return _thread_local.model


# ---------------------------------------------------------------------------
# Image utilities
# ---------------------------------------------------------------------------

def render_page_as_bgr(page: fitz.Page, zoom: float = RENDER_ZOOM) -> np.ndarray:
    """Render a PDF page to a BGR numpy array at the given zoom level."""
    matrix = fitz.Matrix(zoom, zoom)
    pixmap = page.get_pixmap(matrix=matrix, alpha=False)
    img = np.frombuffer(pixmap.samples, dtype=np.uint8).reshape(
        pixmap.height, pixmap.width, pixmap.n
    )
    color_conv = cv2.COLOR_RGBA2BGR if pixmap.n == 4 else cv2.COLOR_RGB2BGR
    return cv2.cvtColor(img, color_conv)


def bgr_to_data_url(img: np.ndarray, jpeg_quality: int = 85) -> str:
    """JPEG-encode a BGR image and return a base64 data URL."""
    success, buf = cv2.imencode(
        ".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality]
    )
    if not success:
        raise RuntimeError("JPEG encoding failed")
    b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


# ---------------------------------------------------------------------------
# Layout detection
# ---------------------------------------------------------------------------

def detect_blocks(img: np.ndarray, page_num: int) -> list[dict]:
    """
    Run YOLO layout detection on a page image and return a list of block dicts.
    Each dict contains position, label, confidence, and the cropped image.
    """
    model = get_yolo_model()
    result = model.predict(img, imgsz=YOLO_IMG_SIZE, conf=YOLO_CONF, device="cpu")[0]

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return []

    names = getattr(result, "names", {})
    h, w = img.shape[:2]
    blocks = []

    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().astype(int).tolist()

        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(x2, w), min(y2, h)

        if x2 <= x1 or y2 <= y1:
            continue

        cls_id = int(boxes.cls[i].item()) if boxes.cls is not None else -1
        label  = names.get(cls_id, f"class_{cls_id}") if isinstance(names, dict) else str(cls_id)
        conf   = float(boxes.conf[i].item()) if boxes.conf is not None else None

        blocks.append({
            "page": page_num,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "label": label,
            "conf": conf,
            "crop": img[y1:y2, x1:x2].copy(),
        })
        cv2.imwrite(f"debug_page{page_num}_block{i+1}_{label}.jpg", blocks[-1]["crop"])  # Debug: save each block crop
        # print(f"    Detected block: {label} (conf={conf:.2f}) at ({x1},{y1},{x2},{y2})")
    return blocks[:MAX_BLOCKS]


# ---------------------------------------------------------------------------
# LLM prompt construction
# ---------------------------------------------------------------------------

SYSTEM_INSTRUCTIONS = """\
You are converting a PDF page into clean Markdown.

You will receive:
  1. The full page image
  2. A set of cropped layout regions detected from that page

Reading order:
  - Do NOT rely on the order in which cropped regions are supplied.
  - Infer the correct reading order from the full page image and the bounding boxes.
  - For multi-column layouts, follow column-by-column reading order.
  - Use the full page as the primary source for layout understanding.
  - Use the crops for detailed text extraction.

Output rules:
  1. Output clean Markdown only — no prose preamble.
  2. Use #/##/### for headings.
  3. Render tables as Markdown tables where possible.
  4. For figures/charts/images, add a detailed placeholder like [Figure: ...] including an in-depth summary without altering the original content.
  5. Skip decorative items, page numbers, and repeated headers/footers.
  6. Do not hallucinate missing text.
  7. Keep uncertain text minimal and faithful to what is visible.\
"""


def build_prompt_content(page_img: np.ndarray, blocks: list[dict], page_num: int) -> list[dict]:
    """Build the multimodal content list for the LLM API call."""
    content = [
        {"type": "input_text", "text": SYSTEM_INSTRUCTIONS},
        {"type": "input_text", "text": f"Page number: {page_num}\nDetected blocks: {len(blocks)}"},
        {"type": "input_text", "text": "Full page image:"},
        {"type": "input_image", "image_url": bgr_to_data_url(page_img)},
        {
            "type": "input_text",
            "text": (
                "Detected blocks follow. Their order is arbitrary — "
                "use bbox coordinates + the full page to determine reading order."
            ),
        },
    ]

    for idx, block in enumerate(blocks, start=1):
        content.append({
            "type": "input_text",
            "text": (
                f"Block {idx}  label={block['label']}  "
                f"conf={block['conf']:.2f}  "
                f"bbox=({block['x1']},{block['y1']},{block['x2']},{block['y2']})"
            ),
        })
        content.append({"type": "input_image", "image_url": bgr_to_data_url(block["crop"])})

    return content


# ---------------------------------------------------------------------------
# Per-page processing
# ---------------------------------------------------------------------------

def process_page(page_num: int, pdf_path: str) -> tuple[int, str]:
    doc  = fitz.open(pdf_path)
    page = doc[page_num - 1]
    page_img = render_page_as_bgr(page)
    cv2.imwrite(f"debug_page_{page_num}.jpg", page_img)  # Debug: save rendered page image
    doc.close()

    blocks = detect_blocks(page_img, page_num)
    cv2.imwrite(blocks)  # Debug: save page image with detected blocks
    if not blocks:
        return page_num, f"\n\n## Page {page_num}\n\n[No layout blocks detected]\n"

    content = build_prompt_content(page_img, blocks, page_num)
    # print(content[1]["text"])  # Log page number and block count
    response = client.responses.create(
        model=MODEL_NAME,
        input=[{"role": "user", "content": content}],
    )

    markdown = response.output_text.strip()
    return page_num, f"\n\n## Page {page_num}\n\n{markdown}\n"


# ---------------------------------------------------------------------------
# Run (directly in notebook)
# ---------------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

with fitz.open(PDF_PATH) as doc:
    total_pages = len(doc)

print(f"Processing {total_pages} pages with {MAX_WORKERS} workers...")

results: dict[int, str] = {}
errors:  list[tuple[int, str]] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(process_page, page_num, PDF_PATH): page_num
        for page_num in range(1, total_pages + 1)
    }

    for future in as_completed(futures):
        page_num = futures[future]
        try:
            pnum, page_md = future.result()
            results[pnum] = page_md
            print(f"  ✓ page {pnum}")
        except Exception as exc:
            errors.append((page_num, str(exc)))
            print(f"  ✗ page {page_num}: {exc}")

# Write output
output_path = Path(OUTPUT_DIR) / "document.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(f"# {Path(PDF_PATH).name}\n")
    for p in range(1, total_pages + 1):
        f.write(results.get(p, f"\n\n## Page {p}\n\n[Extraction failed]\n"))

print(f"\nSaved → {output_path}")

if errors:
    print(f"\n{len(errors)} page(s) failed:")
    for page_num, err in errors:
        print(f"  Page {page_num}: {err}")

Processing 5 pages with 16 workers...





0: 1024x736 2 titles, 7 plain texts, 1 figure, 1 figure_caption, 1 isolate_formula, 1 formula_caption, 11282.9ms
Speed: 69.7ms preprocess, 11282.9ms inference, 4.1ms postprocess per image at shape (1, 3, 1024, 736)
  ✗ page 2: imwrite() missing 1 required positional argument: 'img'
0: 1024x736 1 title, 25 plain texts, 11844.5ms
Speed: 40.5ms preprocess, 11844.5ms inference, 9.0ms postprocess per image at shape (1, 3, 1024, 736)
  ✗ page 5: imwrite() missing 1 required positional argument: 'img'
0: 1024x736 3 titles, 8 plain texts, 2 abandons, 1 figure, 1 figure_caption, 12155.4ms
Speed: 38.9ms preprocess, 12155.4ms inference, 4.3ms postprocess per image at shape (1, 3, 1024, 736)
  ✗ page 1: imwrite() missing 1 required positional argument: 'img'
0: 1024x736 3 titles, 5 plain texts, 1 figure, 1 figure_caption, 1 table, 1 table_caption, 12589.5ms
Speed: 178.3ms preprocess, 12589.5ms inference, 16.6ms postprocess per image at shape (1, 3, 1024, 